# Subcellular fractions vs Total: shape of the log2 CPM distributions, compression and library complexity

HCT116 cells: RNA of FACS-sorted subcellular fractions and Total RNA, sequenced with ONT (`nanoflowz` pipeline).

We leverage on the following concept: If every mRNA had the same share in a fraction, its log2 CPM would equal that in Total, and a scatter of fraction vs Total would follow a line of slope 1. 
A slope below 1 ("buffering", compression of the mRNA concentrations in a fraction relative to Total) can also come from how the fraction library was made. 
The question here is whether the compression follows the complexity of the libraries, i.e. the number of distinct RNA molecules they were made from.

Data (`nanoflowz` runs):

| data set | run directory under GROUP folder on sciCORE | libraries |
|:---|:---|:---|
| Total | `APA_AI_data/wf_runs/v2p0_ONT_HCT116_v1` | wt, siCtrl, siCPSF6; 3 biological replicates each. **siCtrl is the Total reference** |
| ER/TIS libraries | `APA_localization/wf_runs/v2p0_ONT_FACSorganelles_ER_TIS_PEX_Golgi_Aleksei_v1`, flow cell `p40713_o43608_1` | Free, Org2, ER (CANX), TIS (ZFP36L1); siCtrl and siCPSF5 |
| Golgi/PEX libraries | same run, flow cell `p40713_o43608_2` | Free, Org2, Golgi (GOLGA2), PEX (PEX14); siCtrl and siCPSF5 |
| End/TJ | `APA_localization/wf_runs/v2p0_ONT_FACSorganelles_End_TJ_Aleksei_v1` | Free, Org2, Endo (endosomes, EEA1), TJ (tight junctions, OCLN); siCtrl and siCPSF5 |
| input amount control | `APA_localization/wf_runs/v2p0_ONT_FACSorganelles_Aleksei_v1` | siCtrl Total from 1000 ng (barcode01) and from a 100 ng aliquot (barcode02); siCPSF5 Total 1000 ng and 100 ng (barcode03, barcode04) |

Expression is the direct log2 CPM of nuclear protein-coding genes (counts after UMI deduplication, featureCounts), without Sanity and without length correction: ONT reads are full-length molecules, one count per molecule. Replicates of the fractions and of Total come from different experiments and are never paired: every fraction library is compared with the pooled siCtrl Total.

Sections:
1. samples and read QC along the pipeline (`master_read_tracking_stats.tsv`);
2. nuclear contamination, from nuclear marker RNAs;
3. densities of log2 CPM, fraction vs Total, and the principal-axis (PC1) slope;
4. sequencing depth: the slope expected from depth alone;
5. library complexity (Lander–Waterman, from reads before and after UMI deduplication) and its association with the slope;
6. control: Total RNA library from 1000 ng vs 100 ng of input.

In [ ]:
import warnings
warnings.simplefilter('ignore')

import os
import csv
import glob

import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.stats import spearmanr
from scipy.optimize import brentq

from dotenv import load_dotenv

import seaborn as sns; sns.set()
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, HTML
%matplotlib inline

In [ ]:
load_dotenv("APA_localization.scicore.env", override=True)
subdirs = {"wf_runs_dir": os.getenv("WF_RUNS_DIR"), "figures_dir": os.getenv("FIGURES_DIR"), "tables_dir": os.getenv("TABLES_DIR"),
           "lab_group_dir": os.getenv("LAB_GROUP_DIR")}

ANALYSIS = "ONT_organelles_vs_Total"
FIGS = os.path.join(subdirs["figures_dir"], "gene_expression_analysis", ANALYSIS)
TABLES = os.path.join(subdirs["tables_dir"], "gene_expression_analysis", ANALYSIS)
for d in (FIGS, TABLES):
    os.makedirs(d, exist_ok=True)

# nanoflowz output directories
RUNS = {"Total": os.path.join(subdirs["lab_group_dir"], "APA_AI_data", "wf_runs", "v2p0_ONT_HCT116_v1", "results"),
        "ER_TIS_Golgi_PEX": os.path.join(subdirs["wf_runs_dir"], "v2p0_ONT_FACSorganelles_ER_TIS_PEX_Golgi_Aleksei_v1", "results_alignment_with_SpikeIns"),
        "End_TJ": os.path.join(subdirs["wf_runs_dir"], "v2p0_ONT_FACSorganelles_End_TJ_Aleksei_v1", "results_alignment_with_SpikeIns"),
        "input_control": os.path.join(subdirs["wf_runs_dir"], "v2p0_ONT_FACSorganelles_Aleksei_v1", "results_alignment_with_SpikeIns")}
# annotation used by the Total run; the fraction runs used the same annotation plus spike-ins (ERCC, removed below)
GTF = os.path.join(subdirs["lab_group_dir"], "Genomes", "homo_sapiens", "hg38_v42", "gencode.v42.basic.annotation.gtf")

def figure_caption(path, formats=("png", "pdf")):
    # file name of a saved figure, shown with the plot; the links point to the full-size files relative to the
    # HTML report (WF/scripts/make_html_reports.py copies them to <report>/figures/, same subfolders as FIGURES_DIR)
    path = os.path.splitext(path)[0] if path.endswith((".png", ".pdf")) else path
    rel = os.path.relpath(path, subdirs["figures_dir"])
    links = " | ".join(f'<a href="figures/{rel}.{f}" target="_blank">{f}</a>' for f in formats)
    display(HTML(f'<p><b>Figure:</b> <code>{os.path.basename(path)}</code> &nbsp;({links})</p>'))

def savefig(fig, name, formats=("png", "pdf")):
    for fmt in formats:
        fig.savefig(os.path.join(FIGS, f"{name}.{fmt}"), bbox_inches="tight", dpi=300)
    figure_caption(os.path.join(FIGS, name), formats)

def save_tsv(df, name, index=False):
    df.to_csv(os.path.join(TABLES, name), sep="\t", header=True, index=index, quoting=csv.QUOTE_NONE)
    return os.path.join(TABLES, name)

## Samples

- **Total**: barcode → condition and replicate as in the bulk RNA-seq (R3 and R4 renamed to R2 and R3).
- **ER/TIS and Golgi/PEX libraries**: library names from the pooling sheets (`CANX_ZFP36L1_24_libraries_pooling.tsv` for flow cell `o43608_1`, `GOLGA2_PEX14_24_libraries_pooling.tsv` for `o43608_2`; sample S*n* is barcode *n*). The replicates are the dates of the preparations; `si5` is siCPSF5, which shortens 3' UTRs. Barcodes not in the sheets have almost no reads and are kept only in the QC.
- **End/TJ libraries**: library names from the pooling table of that run (sample S*n* is barcode *n*); Eea1 (endosome marker) is named Endo, Occl (tight junction marker) TJ. Three preparations (11.09, 22.09, 25.09), each with Free, Org2, Endo and TJ in siCtrl and siCPSF5.
- **Input amount control**: barcode01 and barcode02 are siCtrl Total from 1000 ng and from a 100 ng aliquot of the same RNA; barcode03 and barcode04, labelled `siCPSF5 Total` and `siCPSF5 Total-LOW` in `ONT_analysis.ipynb`, are taken as the same design for siCPSF5. The other libraries of that run (an earlier pilot) are kept only in the QC.

In [ ]:
# Total (APA_AI_data): barcode -> condition, replicate
TOTAL_LIBRARIES = {"barcode01": ("wt", "R1"), "barcode02": ("siCtrl", "R1"), "barcode03": ("siCPSF6", "R1"),
                   "barcode04": ("wt", "R2"), "barcode05": ("siCtrl", "R2"), "barcode06": ("siCPSF6", "R2"),
                   "barcode07": ("wt", "R3"), "barcode08": ("siCtrl", "R3"), "barcode09": ("siCPSF6", "R3")}
# pooling sheets, S<n> = barcode<n>: "date condition fraction"
POOLING_SHEETS = {
    "p40713_o43608_1": {1: "14.07.25 siCTRL Free", 2: "14.07.25 siCTRL Org2", 3: "14.07.25 siCTRL CANX", 4: "14.07.25 siCTRL TIS",
                        5: "14.07.25 si5 Free", 6: "14.07.25 si5 Org2", 7: "14.07.25 si5 CANX", 8: "14.07.25 si5 TIS",
                        9: "18.07.25 siCTRL Free", 10: "18.07.25 siCTRL Org2", 11: "18.07.25 siCTRL CANX", 12: "18.07.25 siCTRL TIS",
                        13: "18.07.25 si5 Free", 14: "18.07.25 si5 Org2", 15: "18.07.25 si5 CANX", 16: "18.07.25 si5 TIS",
                        22: "23.01.26 si5 Org2"},
    "p40713_o43608_2": {1: "29.09 siCTRL Free", 2: "29.09 siCTRL Org2", 3: "29.09 siCTRL Golgi", 4: "29.09 siCTRL Perox",
                        5: "29.09 si5 Free", 6: "29.09 si5 Org2", 7: "29.09 si5 Golgi", 8: "29.09 si5 Perox",
                        9: "2.10 siCTRL Free", 10: "2.10 siCTRL Org2", 11: "2.10 siCTRL Golga2", 12: "2.10 siCTRL Pex14",
                        13: "2.10 si5 Free", 14: "2.10 si5 Org2", 15: "2.10 si5 Golga2", 16: "2.10 si5 Pex14"}}
BATCH_OF_FLOWCELL = {"p40713_o43608_1": "ER/TIS libraries", "p40713_o43608_2": "Golgi/PEX libraries"}
# End/TJ: barcode -> "date condition fraction" (pooling table of the End/TJ run, S<n> = barcode<n>)
END_TJ_LIBRARIES = {"barcode01": "11.09 siCTRL Free", "barcode02": "11.09 siCTRL Org2", "barcode03": "11.09 siCTRL Eea1", "barcode04": "11.09 siCTRL Occl",
                    "barcode05": "11.09 si5 Free", "barcode06": "11.09 si5 Org2", "barcode07": "11.09 si5 Eea1", "barcode08": "11.09 si5 Occl",
                    "barcode09": "22.09 siCTRL Free", "barcode10": "22.09 siCTRL Org2", "barcode11": "22.09 siCTRL Eea1", "barcode12": "22.09 siCTRL Occl",
                    "barcode13": "22.09 si5 Free", "barcode14": "22.09 si5 Org2", "barcode15": "22.09 si5 Eea1", "barcode16": "22.09 si5 Occl",
                    "barcode17": "25.09 siCTRL Free", "barcode18": "25.09 siCTRL Org2", "barcode19": "25.09 siCTRL Eea1", "barcode20": "25.09 siCTRL Occl",
                    "barcode21": "25.09 si5 Free", "barcode22": "25.09 si5 Org2", "barcode23": "25.09 si5 Eea1", "barcode24": "25.09 si5 Occl"}
# input amount control and the earlier pilot libraries of the same run (labels from ONT_analysis.ipynb)
INPUT_CONTROL = {"barcode01": ("siCtrl", "Total 1000 ng"), "barcode02": ("siCtrl", "Total 100 ng"),
                 "barcode03": ("siCPSF5", "Total 1000 ng"), "barcode04": ("siCPSF5", "Total 100 ng")}
PILOT = {"barcode05": "siCtrl Organelles", "barcode06": "siCPSF5 Organelles", "barcode07": "siCtrl TIS", "barcode08": "siCPSF5 TIS",
         "barcode09": "siCtrl ER", "barcode10": "siCPSF5 ER", "barcode11": "siCtrl TISnew", "barcode12": "siCPSF5 TISnew"}

FRACTION_NAMES = {"Free": "Free", "Org2": "Org2", "CANX": "ER", "TIS": "TIS", "Golgi": "Golgi", "Golga2": "Golgi", "Perox": "PEX", "Pex14": "PEX", "Eea1": "Endo", "Occl": "TJ"}
CONDITION_NAMES = {"siCTRL": "siCtrl", "si5": "siCPSF5"}
FRACTION_ORDER = ["Total", "Free", "Org2", "ER", "TIS", "Golgi", "PEX", "Endo", "TJ", "End/TJ (unlabelled)", "Total 1000 ng", "Total 100 ng"]
FRACTION_COLORS = {"Total": "black", "Free": "tab:blue", "Org2": "tab:orange", "ER": "tab:green", "TIS": "tab:red", "Golgi": "tab:purple",
                   "PEX": "tab:brown", "End/TJ (unlabelled)": "grey", "Total 1000 ng": "darkslategrey", "Total 100 ng": "tab:cyan",
                   "Endo": "tab:pink", "TJ": "tab:olive", "other": "lightgrey"}
CONDITION_MARKERS = {"siCtrl": "o", "siCPSF5": "^", "wt": "s", "siCPSF6": "D", "unknown": "X"}
BATCH_ORDER = ["HCT116 Total", "ER/TIS libraries", "Golgi/PEX libraries", "End/TJ libraries", "input amount control"]

def parse_library_name(name):
    date, cond, frac = name.split()
    return CONDITION_NAMES.get(cond, cond), FRACTION_NAMES.get(frac, frac), date

def gene_count_files(source):
    files = sorted(glob.glob(os.path.join(RUNS[source], "gene_assignments", "*.featureCounts.txt")))
    return {os.path.basename(f).replace(".featureCounts.txt", ""): f for f in files}

rows = []
for source in RUNS:
    for sample_id in gene_count_files(source):
        r = {"source": source, "sample_id": sample_id, "condition": "unknown", "replicate": "", "use": "QC only"}
        if source == "Total":
            r["condition"], r["replicate"] = TOTAL_LIBRARIES[sample_id]
            r.update(batch="HCT116 Total", fraction="Total", use="Total")
        elif source == "ER_TIS_Golgi_PEX":
            barcode, flowcell = sample_id.split("_", 1)
            r["batch"] = BATCH_OF_FLOWCELL[flowcell]
            name = POOLING_SHEETS[flowcell].get(int(barcode.replace("barcode", "")))
            if name:
                r["condition"], r["fraction"], r["replicate"] = parse_library_name(name)
                r["use"] = "fraction"
            else:
                r["fraction"] = "other"
        elif source == "End_TJ":
            r["batch"] = "End/TJ libraries"
            if sample_id in END_TJ_LIBRARIES:
                r["condition"], r["fraction"], r["replicate"] = parse_library_name(END_TJ_LIBRARIES[sample_id])
                r["use"] = "fraction"
            else:
                r.update(fraction="End/TJ (unlabelled)", replicate=sample_id, use="fraction, unlabelled")
        else:
            r["batch"] = "input amount control"
            if sample_id in INPUT_CONTROL:
                r["condition"], r["fraction"] = INPUT_CONTROL[sample_id]
                r["use"] = "input control"
            else:
                r.update(fraction="other", replicate=PILOT[sample_id])
        rows.append(r)
libs = pd.DataFrame(rows)
short_batch = {"HCT116 Total": "Total", "ER/TIS libraries": "ER/TIS", "Golgi/PEX libraries": "Golgi/PEX", "End/TJ libraries": "End/TJ",
               "input amount control": "input ctrl"}
libs["library"] = [f"{short_batch[b]}: {s}" if f in ("other", "End/TJ (unlabelled)") else
                   (f"{short_batch[b]}: {f} {c} {r}" if b != "input amount control" else f"{f} {c}")
                   for b, f, c, r, s in zip(libs["batch"], libs["fraction"], libs["condition"], libs["replicate"], libs["sample_id"])]
# group of a fraction library: batch, condition and fraction (its replicates are pooled)
libs["group"] = np.where(libs["use"] == "fraction", libs["batch"].map(short_batch) + ": " + libs["fraction"] + " " + libs["condition"], "")
libs["batch"] = pd.Categorical(libs["batch"], BATCH_ORDER, ordered=True)
libs["fraction_rank"] = libs["fraction"].map({f: i for i, f in enumerate(FRACTION_ORDER)}).fillna(len(FRACTION_ORDER))
libs = libs.sort_values(["batch", "fraction_rank", "condition", "replicate", "sample_id"]).drop(columns="fraction_rank").reset_index(drop=True)
libs = libs.set_index("library", drop=False).rename_axis(None)
assert libs.index.is_unique
save_tsv(libs, "libraries.tsv")
display(libs.groupby(["batch", "use", "fraction", "condition"], observed=True).size().rename("libraries").reset_index())

## Read QC along the pipeline

From `master_read_tracking_stats.tsv` of each run (distinct read IDs at each step; mapped reads are uniquely and multi-mapped reads):
- `basecalled`: reads from Dorado;
- `mapped`: reads aligned by minimap2;
- `poly(A)`: mapped reads with a poly(A) tail at a cleavage site (SCINPAS), the reads used in the analysis;
- `before dedup` and `after dedup`: mapped reads before and after UMI deduplication (`umi_tools dedup`), the inputs of the library complexity below.

From the gene counts (featureCounts, after deduplication; a read overlapping several genes counts for each): the share of gene assignments on nuclear protein-coding genes, on genes of the mitochondrial genome (`chrM`) and on all other genes.

In [ ]:
STEPS = {"basecalled": "01_dorado_basecall", "mapped": "03_aligned_with_minimap2", "poly(A)": "06_extracted_polyA_reads",
         "before dedup": "07_appended_polyA_tails", "after dedup": "08_umi_deduped"}

def read_tracking(source):
    df = pd.read_csv(os.path.join(RUNS[source], "QC", "master_read_tracking_stats.tsv"), sep="\t")
    df["mapped_reads"] = df["um_reads"] + df["mm_reads"]
    wide = df.pivot(index="sample_id", columns="step", values="mapped_reads")
    out = pd.DataFrame({k: wide[s] for k, s in STEPS.items()})
    out["basecalled"] = df.loc[df["step"] == STEPS["basecalled"]].set_index("sample_id")["total_reads"]
    return out.assign(source=source).reset_index()

tracking = pd.concat([read_tracking(s) for s in RUNS], ignore_index=True)
qc = libs[["library", "source", "sample_id", "batch", "fraction", "condition", "replicate", "use", "group"]].merge(
    tracking, on=["source", "sample_id"], how="left").set_index("library", drop=False).rename_axis(None)

# gene annotation (genes only, read in chunks)
genes = []
for chunk in pd.read_csv(GTF, sep="\t", comment="#", header=None, usecols=[0, 2, 8], names=["chrom", "type", "attr"], chunksize=500_000):
    genes.append(chunk.loc[chunk["type"] == "gene"])
genes = pd.concat(genes)
genes = pd.DataFrame({"chrom": genes["chrom"].to_numpy(),
                      "gene_type": genes["attr"].str.extract(r'gene_type "([^"]+)"')[0].to_numpy(),
                      "gene_name": genes["attr"].str.extract(r'gene_name "([^"]+)"')[0].to_numpy()},
                     index=genes["attr"].str.extract(r'gene_id "([^"]+)"')[0].to_numpy())

# gene counts after deduplication, genes of the annotation (spike-ins removed), one column per library
counts = {}
for source in RUNS:
    for sample_id, f in gene_count_files(source).items():
        fc = pd.read_csv(f, sep="\t", comment="#", index_col=0).iloc[:, -1]
        counts[libs.loc[(libs["source"] == source) & (libs["sample_id"] == sample_id), "library"].iloc[0]] = fc.reindex(genes.index)
counts = pd.DataFrame(counts)[libs["library"]].fillna(0).astype(np.int64)

nuclear_pc = genes.index[(genes["gene_type"] == "protein_coding") & (genes["chrom"] != "chrM")]
mito = genes.index[genes["chrom"] == "chrM"]
assigned = counts.sum()
qc["gene assignments"] = assigned
qc["share nuclear protein-coding"] = counts.loc[nuclear_pc].sum() / assigned
qc["share chrM"] = counts.loc[mito].sum() / assigned
qc["share other genes"] = 1 - qc["share nuclear protein-coding"] - qc["share chrM"]
qc["mapped / basecalled"] = qc["mapped"] / qc["basecalled"]
qc["poly(A) / mapped"] = qc["poly(A)"] / qc["mapped"]
qc["kept by dedup"] = qc["after dedup"] / qc["before dedup"]
save_tsv(qc, "QC.read_tracking_and_gene_assignments.tsv")

QC_COLS = ["basecalled", "mapped / basecalled", "poly(A) / mapped", "kept by dedup", "after dedup", "share nuclear protein-coding", "share chrM"]
fmt = {c: "{:,.0f}" for c in ("basecalled", "after dedup")} | {c: "{:.3f}" for c in QC_COLS if c not in ("basecalled", "after dedup")}
display(qc.loc[qc["use"] != "QC only", QC_COLS].style.format(fmt))
print("QC only (not in the pooling sheets or earlier pilot):")
display(qc.loc[qc["use"] == "QC only", ["basecalled", "after dedup"]].style.format("{:,.0f}"))

In [ ]:
sns.set(font_scale=0.9, style="ticks")
qc_plot = qc.loc[qc["use"] != "QC only"]
batches = [b for b in BATCH_ORDER if (qc_plot["batch"] == b).any()]
fig, axes = plt.subplots(1, len(batches), figsize=(3.4 * len(batches), 3.6), sharey=True)
for ax, b in zip(axes, batches):
    for lib, r in qc_plot.loc[qc_plot["batch"] == b].iterrows():
        ax.plot(range(len(STEPS)), r[list(STEPS)].astype(float), color=FRACTION_COLORS.get(r["fraction"], "grey"),
                marker=CONDITION_MARKERS[r["condition"]], ms=4, lw=0.8, alpha=0.8)
    ax.set_xticks(range(len(STEPS)))
    ax.set_xticklabels(list(STEPS), rotation=45, ha="right")
    ax.set(yscale="log", title=b)
axes[0].set_ylabel("reads")
handles = [plt.Line2D([], [], color=FRACTION_COLORS[f], lw=2, label=f) for f in FRACTION_ORDER if (qc_plot["fraction"] == f).any()] + \
          [plt.Line2D([], [], color="grey", marker=m, ls="", label=c) for c, m in CONDITION_MARKERS.items() if (qc_plot["condition"] == c).any()]
axes[-1].legend(handles=handles, fontsize=7, frameon=False, loc="upper left", bbox_to_anchor=(1.02, 1.0))
sns.despine()
fig.tight_layout()
savefig(fig, "QC.reads_along_pipeline")
plt.show()

# per library: composition of the gene assignments, and the shares kept by the poly(A) selection and by deduplication.
# Libraries on the y axis; their names are colored by the fraction, and the marker left of each name shows fraction and condition.
n = len(qc_plot)
y = np.arange(n)
fig, axes = plt.subplots(1, 3, figsize=(11, max(6, 0.16 * n)), sharey=True, gridspec_kw={"width_ratios": [0.15, 4, 4], "wspace": 0.08})
ann = axes[0]
for m in set(qc_plot["condition"]):
    sel = (qc_plot["condition"] == m).to_numpy()
    ann.scatter(np.zeros(sel.sum()), y[sel], c=[FRACTION_COLORS.get(f, "grey") for f in qc_plot.loc[sel, "fraction"]],
                marker=CONDITION_MARKERS[m], s=22, lw=0)
ann.set(xlim=(-0.5, 0.5), xticks=[], ylim=(n - 0.5, -0.5))
ann.set_yticks(y)
ann.set_yticklabels(qc_plot["library"], fontsize=6.5)
for t, f in zip(ann.get_yticklabels(), qc_plot["fraction"]):
    t.set_color(FRACTION_COLORS.get(f, "black"))
ann.tick_params(axis="y", length=0)
for s in ann.spines.values():
    s.set_visible(False)

left = np.zeros(n)
for col, color in (("share nuclear protein-coding", "tab:blue"), ("share chrM", "tab:red"), ("share other genes", "lightgrey")):
    axes[1].barh(y, qc_plot[col], left=left, color=color, height=0.85, label=col.replace("share ", ""))
    left += qc_plot[col].to_numpy()
axes[1].set(xlabel="share of gene assignments", xlim=(0, 1))
axes[1].legend(fontsize=7, frameon=False, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=3)
for col, color in (("poly(A) / mapped", "tab:purple"), ("kept by dedup", "tab:green"), ("mapped / basecalled", "black")):
    axes[2].scatter(qc_plot[col], y, color=color, s=14, label=col)
axes[2].set(xlabel="fraction of reads", xlim=(0, 1.02), xticks=[0.2, 0.4, 0.6, 0.8, 1.0])
axes[2].legend(fontsize=7, frameon=False, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=3)
for ax in axes[1:]:
    ax.tick_params(axis="y", left=False)
    ax.grid(axis="x", color="0.9", lw=0.6)
    ax.set_axisbelow(True)
    for b in batches[1:]:
        ax.axhline(np.where(qc_plot["batch"] == b)[0][0] - 0.5, color="grey", lw=0.6)
fig.legend(handles=handles, fontsize=7, frameon=False, loc="upper left", bbox_to_anchor=(0.91, 0.88))
sns.despine(fig=fig, left=True)
ann.spines["bottom"].set_visible(False)
savefig(fig, "QC.composition_polyA_dedup.per_library")
plt.show()


#### Interpretation: read QC

- **Depth.** The Total libraries have 20–26 M basecalled reads and 12–18 M after deduplication. The fraction libraries have 10 to 100 times fewer: 1.2–5.0 M basecalled in the Golgi/PEX libraries, 0.1–2.7 M in End/TJ, and mostly below 0.7 M in the ER/TIS libraries (one ER library has 12 M).
- **Mitochondrial reads.** Their share differs between preparations far more than between fractions. Free libraries have 2% chrM reads in the Golgi/PEX batch, but up to 96% in the ER/TIS batch. The ER/TIS libraries of 14.07.25 (Free) and 18.07.25 (TIS) are almost entirely mitochondrial and keep fewer than 10,000 reads on nuclear genes. PEX libraries have 34–60% chrM reads, TJ libraries 40–64%, Endo 10–15%, and the End/TJ Free libraries 3%. All expression values below are therefore computed over nuclear protein-coding genes only.
- **Poly(A) reads.** The share of mapped reads with a poly(A) tail (0.41–0.85) is similar in all data sets.
- **Deduplication.** Deduplication keeps 0.80–0.96 of the reads of the fraction libraries, close to 1 because they are shallow. In the input amount control, it keeps 0.92–0.93 at 1000 ng but only 0.68–0.71 at 100 ng, at similar depth: the 100 ng libraries contain fewer distinct molecules.

## Nuclear contamination

Markers of the nucleus that are detected in the poly(A) reads:
- **MALAT1**, a nuclear-retained lncRNA (nuclear speckles);
- **NEAT1**, a nuclear-retained lncRNA (paraspeckles). Its short isoform NEAT1_1 is made by cleavage and polyadenylation, so its reads in poly(A) data change with knockdowns of cleavage and polyadenylation factors (in Total, $R$ of NEAT1 is 0.26 in siCPSF6 against 1.03 for MALAT1);
- **snoRNAs** (sum over snoRNA genes), which stay in the nucleus; in poly(A)-selected reads they come from unprocessed precursors and host-gene transcripts.

Per library, the reads of a marker per million reads on nuclear protein-coding genes, relative to the pooled siCtrl Total: $R = 1$ means as much nuclear RNA as in whole cells. Normalizing to nuclear protein-coding reads, and not to all reads, removes the large differences in the share of mitochondrial reads between fractions (QC above). Intervals are exact Poisson 95% intervals of the marker count (the Total reference has millions of reads, its error is negligible).

How to read $R$: if a marker is exclusively nuclear, the share of nucleus-derived RNA in a library is $f = R \cdot \phi$, with $\phi$ the share of nuclear RNA among the poly(A) RNA of whole cells (not measured here); a purely nuclear sample has $R = 1/\phi$, a sample free of nuclear RNA $R = 0$. **MALAT1 is used for the estimate** below: it is the most specific marker and does not change with the knockdowns in Total. The implied nuclear share $f$ is shown for $\phi$ = 5% and 10%.

In [ ]:
NUCLEAR_MARKERS = {"MALAT1": genes.index[genes["gene_name"] == "MALAT1"], "NEAT1": genes.index[genes["gene_name"] == "NEAT1"],
                   "snoRNAs": genes.index[genes["gene_type"] == "snoRNA"]}
total_ref = libs.index[(libs["fraction"] == "Total") & (libs["condition"] == "siCtrl")]
nuc_reads = counts.loc[nuclear_pc].sum()
ref_nuc = nuc_reads[total_ref].sum()

nuc_rows = []
for m, ids in NUCLEAR_MARKERS.items():
    k = counts.loc[ids].sum()
    ref_rate = k[total_ref].sum() / ref_nuc
    lo = np.where(k > 0, stats.chi2.ppf(0.025, 2 * k) / 2, 0.0)
    hi = stats.chi2.ppf(0.975, 2 * k + 2) / 2
    nuc_rows.append(pd.DataFrame({"library": counts.columns, "marker": m, "marker_reads": k.to_numpy(), "nuclear_pc_reads": nuc_reads.to_numpy(),
                                  "per_million_nuclear_pc": (k / nuc_reads * 1e6).to_numpy(), "R": (k / nuc_reads / ref_rate).to_numpy(),
                                  "R_low": lo / nuc_reads.to_numpy() / ref_rate, "R_high": hi / nuc_reads.to_numpy() / ref_rate}))
nuc_df = pd.concat(nuc_rows, ignore_index=True)
nuc_wide = nuc_df.pivot(index="library", columns="marker", values="R").reindex(libs.index)
for m in NUCLEAR_MARKERS:
    qc[f"R {m}"] = nuc_wide[m]
qc["log2 R MALAT1"] = np.log2(qc["R MALAT1"].clip(lower=1e-2))
PHI = (0.05, 0.10)
for phi in PHI:
    qc[f"nuclear share, phi={phi:.2f}"] = (qc["R MALAT1"] * phi).clip(upper=1)
save_tsv(nuc_df.merge(libs[["library", "batch", "fraction", "condition", "replicate"]], on="library"), "nuclear_markers.per_library.tsv")

MIN_NUC_READS = 20_000   # below, the intervals of R span an order of magnitude
show = qc.loc[(qc["use"] != "QC only") & (nuc_reads.reindex(qc.index) >= MIN_NUC_READS)]
NUC_COLS = ["R MALAT1", "R NEAT1", "R snoRNAs"] + [f"nuclear share, phi={phi:.2f}" for phi in PHI]
display(show[["batch", "fraction", "condition", "replicate"] + NUC_COLS].round(2))
print(f"not shown (< {MIN_NUC_READS:,} reads on nuclear protein-coding genes):",
      ", ".join(qc.index[(qc["use"] != "QC only") & (nuc_reads.reindex(qc.index) < MIN_NUC_READS)]))

sns.set(font_scale=0.9, style="ticks")
fig, ax = plt.subplots(figsize=(max(12, 0.2 * len(show)), 4.4))
x = np.arange(len(show))
for j, (m, color) in enumerate(zip(NUCLEAR_MARKERS, ("tab:red", "tab:orange", "tab:purple"))):
    d = nuc_df.set_index("library").loc[nuc_df["marker"].eq(m).to_numpy()].reindex(show.index)
    y = np.log2(d["R"].clip(lower=1e-2))
    ax.errorbar(x + (j - 1) * 0.22, y, yerr=[y - np.log2(d["R_low"].clip(lower=1e-2)), np.log2(d["R_high"].clip(lower=1e-2)) - y],
                fmt="o", ms=3, lw=0.8, color=color, label=m)
ax.axhline(0, color="grey", lw=0.8, ls="--")
for b in batches[1:]:
    ax.axvline(np.where(show["batch"] == b)[0][0] - 0.5, color="grey", lw=0.6)
ax.set_xticks(x)
ax.set_xticklabels(show.index, rotation=90, fontsize=6)
for t, f in zip(ax.get_xticklabels(), show["fraction"]):
    t.set_color(FRACTION_COLORS.get(f, "black"))
ax.set_ylabel("log2 R (relative to Total siCtrl)")
ax.legend(fontsize=7, frameon=False, loc="upper left", bbox_to_anchor=(1.0, 1.0))
sns.despine()
fig.tight_layout()
savefig(fig, "nuclear_contamination.markers_per_library")
plt.show()

# per fraction and condition: geometric mean of R over the libraries, and the implied nuclear share (MALAT1)
gm = show.loc[show["use"].isin(["Total", "fraction", "input control"])].groupby(["batch", "fraction", "condition"], observed=True)
nuc_group = np.exp(gm[["R MALAT1", "R NEAT1", "R snoRNAs"]].agg(lambda s: np.log(s.clip(lower=1e-2)).mean()))
for phi in PHI:
    nuc_group[f"nuclear share, phi={phi:.2f}"] = (nuc_group["R MALAT1"] * phi).clip(upper=1)
save_tsv(nuc_group.reset_index(), "nuclear_markers.by_fraction.tsv")
display(nuc_group.round(2))

#### Interpretation: nuclear contamination

$R$ of MALAT1 (geometric mean over the replicates; siCtrl / siCPSF5), and the implied nuclear share $f = R \cdot \phi$ for $\phi = 10\%$:

| fraction | $R$ MALAT1 | nuclear share ($\phi$ = 10%) |
|:---|:---|:---|
| Free (all three batches) | 0.08–0.40 / 0.01–0.08 | ≤ 4% |
| TIS | 0.37 / 0.45 | 4% |
| Org2 | 1.0–2.7 / 0.6–1.0 | 6–27% |
| ER | 1.8 / 1.6 | 16–18% |
| Golgi | 4.5 / 2.8 | 28–45% |
| PEX | 8.2 / 4.6 | 46–82% |
| Endo | 2.0 / 1.4 | 14–20% |
| TJ | 7.1 / 3.6 | 36–71% |
| Total (HCT116) | 1.0 by definition | 10% |

- **Free and TIS are essentially free of nuclear RNA.** Org2 and ER contain about as much as whole cells, or up to 3 times more. ER is continuous with the nuclear envelope, so some nuclear material in the ER sort is expected.
- **Golgi, PEX and TJ are strongly contaminated with nuclear RNA, Endo moderately.** In the siCtrl PEX and TJ libraries, MALAT1 per mRNA is 7–8 times higher than in whole cells. With $\phi$ = 5–10% of poly(A) RNA being nuclear in whole cells, 35–80% of the RNA of these libraries would come from nuclei. NEAT1 agrees (9–14 times higher than in whole cells). Endo has 1.4–2 times the MALAT1 of whole cells, about as much as Org2 and ER.
- **siCPSF5 vs siCtrl.** In Org2, Golgi, PEX, Endo and TJ, siCPSF5 libraries have 30–70% of the MALAT1 and NEAT1 of siCtrl (not in ER and TIS). This may be less nuclear carry-over in those sorts, or an effect of the knockdown on how the 3' ends of these RNAs are captured; the data do not distinguish the two. In Total, MALAT1 is unaffected by siCPSF6 while NEAT1 drops 4-fold.
- **snoRNAs do not discriminate between fractions** ($R$ 0.4–0.9 everywhere), so they are not used.
- **The Total libraries of the input amount control** have $R$ 0.2–0.5, less nuclear RNA than the HCT116 Total. The two Total RNA preparations differ.
- **The Free libraries of End/TJ** (barcodes 01, 05, 09, 13, 17, 21) have $R \le 0.24$, like the Free libraries of the other batches.


## Expression: log2 CPM of protein-coding genes

Genes: nuclear protein-coding genes with at least 10 CPM in the pooled siCtrl Total (CPM over nuclear protein-coding genes), the same genes in every comparison. Selecting the genes on Total, and not on the fractions, keeps the same set for every fraction; it truncates the low end of Total only, which, if anything, raises the slopes below (against compression).

Per library, and per group (the replicates of a fraction and condition in one batch, counts summed): log2 CPM over these genes with a pseudocount of 0.5 reads. The Total reference is the sum of the three siCtrl Total libraries.

Libraries with fewer than 100,000 reads on these genes are left out of the per-library analyses (listed below); they still enter the sums of their groups.

Shape statistics per group, as in `pbody_RNA_Spang.SanityVmethod.ipynb`: standard deviation of log2 CPM over genes, share of genes below log2 CPM 0, 5% quantile, median, 99% quantile, Kolmogorov–Smirnov distance to Total.

In [ ]:
pooled_total = counts[total_ref].sum(axis=1)
cpm_total = pooled_total[nuclear_pc] / pooled_total[nuclear_pc].sum() * 1e6
universe = cpm_total.index[cpm_total >= 10]
print(f"genes: {len(universe):,} nuclear protein-coding genes with >= 10 CPM in the pooled siCtrl Total "
      f"(of {len(nuclear_pc):,}; {pooled_total[universe].sum() / pooled_total[nuclear_pc].sum():.1%} of their reads)")

def log2_cpm(cnt, pseudocount=0.5):
    x = cnt.loc[universe] + pseudocount
    return np.log2(x / x.sum() * 1e6)

def pc1_slope(t, p):
    # slope of the first principal axis of the centred (Total, fraction) values
    w, V = np.linalg.eigh(np.cov(np.vstack([t, p])))
    return V[1, -1] / V[0, -1]

qc["reads on genes"] = counts.loc[universe].sum()
MIN_READS = 100_000
fraction_libs = libs.index[libs["use"].isin(["fraction", "fraction, unlabelled"])]
analysed = [l for l in fraction_libs if qc.loc[l, "reads on genes"] >= MIN_READS]
print(f"fraction libraries with >= {MIN_READS:,} reads on these genes: {len(analysed)} of {len(fraction_libs)}; left out:")
display(qc.loc[[l for l in fraction_libs if l not in analysed], ["batch", "fraction", "condition", "replicate", "after dedup", "share chrM", "reads on genes"]])

# groups: replicates of a fraction and condition in one batch, summed
group_libs = libs.loc[libs["group"] != ""].groupby("group", sort=False)["library"].apply(list)
group_info = libs.loc[libs["group"] != ""].groupby("group", sort=False)[["batch", "fraction", "condition"]].first()
group_counts = pd.DataFrame({g: counts[l].sum(axis=1) for g, l in group_libs.items()})

t_ref = log2_cpm(pooled_total).to_numpy()
lib_values = log2_cpm(counts)
group_values = log2_cpm(group_counts)

def shape_stats(p, t=t_ref):
    return {"sd_Total": t.std(), "sd_fraction": p.std(), "frac_below_0_Total": (t < 0).mean(), "frac_below_0_fraction": (p < 0).mean(),
            "q05_Total": np.quantile(t, 0.05), "q05_fraction": np.quantile(p, 0.05), "median_Total": np.median(t), "median_fraction": np.median(p),
            "q99_Total": np.quantile(t, 0.99), "q99_fraction": np.quantile(p, 0.99), "KS_fraction_vs_Total": stats.ks_2samp(p, t).statistic}

group_shape = pd.DataFrame({g: {"n_libraries": len(group_libs[g]), "reads_on_genes": int(group_counts[g].loc[universe].sum()),
                                **shape_stats(group_values[g].to_numpy())} for g in group_counts}).T
group_shape = group_info.join(group_shape)
save_tsv(group_shape.reset_index(names="group"), "fraction_vs_Total.shape_stats.by_group.tsv")
display(group_shape.drop(columns=["sd_Total", "frac_below_0_Total", "q05_Total", "median_Total", "q99_Total"]).round(3))

### Densities of log2 CPM

Total siCtrl (black) and each group (color of the fraction), per batch: rows are the conditions, columns the fractions.

In [ ]:
grid = np.linspace(-4, 16, 300)
LABELLED_BATCHES = [b for b in BATCH_ORDER if (group_info["batch"] == b).any()]

def group_grid(batch):
    gi = group_info.loc[group_info["batch"] == batch]
    fracs = [f for f in FRACTION_ORDER if (gi["fraction"] == f).any()]
    conds = [c for c in CONDITION_MARKERS if (gi["condition"] == c).any()]
    return gi, fracs, conds

for batch in LABELLED_BATCHES:
    gi, fracs, conds = group_grid(batch)
    sns.set(font_scale=0.9, style="ticks")
    fig, axes = plt.subplots(len(conds), len(fracs), figsize=(2.8 * len(fracs), 2.5 * len(conds)), sharex=True, sharey=True, squeeze=False)
    for g, r in gi.iterrows():
        ax = axes[conds.index(r["condition"]), fracs.index(r["fraction"])]
        ax.plot(grid, stats.gaussian_kde(t_ref)(grid), color="black", lw=1.2, label="Total siCtrl")
        ax.plot(grid, stats.gaussian_kde(group_values[g])(grid), color=FRACTION_COLORS[r["fraction"]], lw=1.8, label="fraction")
        ax.set_title(f"{r['fraction']} {r['condition']} (n={len(group_libs[g])})", fontsize=9)
    for ax in axes[-1]:
        ax.set_xlabel("log2 CPM")
    for ax in axes[:, 0]:
        ax.set_ylabel("density")
    axes[0, 0].legend(fontsize=7, frameon=False, loc="upper left")
    fig.suptitle(batch, y=1.02, fontsize=12)
    sns.despine()
    fig.tight_layout()
    savefig(fig, f"fraction_vs_Total.density.{batch.split()[0].replace('/', '_')}")
    plt.show()

### Fraction vs Total per gene: principal axis

As in `pbody_RNA_Spang.SanityVmethod.ipynb`: log2 CPM in the fraction against that in Total, with the principal axis (PC1 of the centred values; orthogonal regression, both axes treated alike) through the means, and the line of slope 1 (dashed). The PC1 slope is below 1 exactly when the fraction varies less over genes than Total, i.e. when it is compressed. Measurement noise adds variance to both axes; the shallow fraction libraries have more of it than Total, which biases the slope upwards, against compression (quantified in the next section).

Reference slopes from Total itself: each siCtrl Total library against the sum of the other two (leave one out), and the wt and siCPSF6 Total libraries against the siCtrl Total.

In [ ]:
def slope_row(lib, x_values, y_values):
    t, p = x_values, y_values
    return {"pc1_slope": pc1_slope(t, p), "pearson_r": np.corrcoef(t, p)[0, 1], "sd_Total": t.std(), "sd_fraction": p.std()}

# per library: fraction vs pooled siCtrl Total
slope_rows = []
for l in analysed:
    slope_rows.append({"library": l, "reads_on_genes": int(qc.loc[l, "reads on genes"]), **slope_row(l, t_ref, lib_values[l].to_numpy())})
# references from Total: siCtrl leave one out, wt and siCPSF6 vs siCtrl
for l in libs.index[libs["fraction"] == "Total"]:
    ref = [r for r in total_ref if r != l]
    x = log2_cpm(counts[ref].sum(axis=1)).to_numpy()
    slope_rows.append({"library": l, "reads_on_genes": int(qc.loc[l, "reads on genes"]), **slope_row(l, x, lib_values[l].to_numpy())})
lib_slopes = libs[["library", "batch", "fraction", "condition", "replicate", "group"]].merge(pd.DataFrame(slope_rows), on="library").set_index("library", drop=False).rename_axis(None)
group_slopes = group_info.join(pd.DataFrame({g: slope_row(g, t_ref, group_values[g].to_numpy()) for g in group_counts}).T)

save_tsv(lib_slopes, "fraction_vs_Total.pc1_slope.per_library.tsv")
save_tsv(group_slopes.reset_index(names="group"), "fraction_vs_Total.pc1_slope.by_group.tsv")
print("PC1 slope per group (replicates summed):")
display(group_slopes.pivot_table(index=["batch", "fraction"], columns="condition", values="pc1_slope", observed=True).round(3))
print("Total references (siCtrl: leave one out; wt, siCPSF6: vs siCtrl):")
display(lib_slopes.loc[lib_slopes["fraction"] == "Total", ["condition", "replicate", "reads_on_genes", "pc1_slope", "pearson_r"]].round(3))

lims = np.array([-1.0, 15.0])
for batch in LABELLED_BATCHES:
    gi, fracs, conds = group_grid(batch)
    sns.set(font_scale=0.9, style="ticks")
    fig, axes = plt.subplots(len(conds), len(fracs), figsize=(2.8 * len(fracs), 2.8 * len(conds)), sharex=True, sharey=True, squeeze=False)
    for g, r in gi.iterrows():
        ax = axes[conds.index(r["condition"]), fracs.index(r["fraction"])]
        p = group_values[g].to_numpy()
        ax.scatter(t_ref, p, s=1.5, alpha=0.25, color=FRACTION_COLORS[r["fraction"]], lw=0, rasterized=True)
        mt, mp, b = t_ref.mean(), p.mean(), pc1_slope(t_ref, p)
        ax.plot(lims, mp + (lims - mt), color="grey", lw=0.8, ls="--")
        ax.plot(lims, mp + b * (lims - mt), color="black", lw=1.2)
        ax.text(0.04, 0.96, f"PC1 slope {b:.2f}\nr {np.corrcoef(t_ref, p)[0, 1]:.2f}", transform=ax.transAxes, va="top", fontsize=8)
        ax.set(xlim=lims, ylim=lims, aspect="equal")
        ax.set_title(f"{r['fraction']} {r['condition']} (n={len(group_libs[g])})", fontsize=9)
    for ax in axes[-1]:
        ax.set_xlabel("log2 CPM, Total siCtrl")
    for ax in axes[:, 0]:
        ax.set_ylabel("log2 CPM, fraction")
    fig.suptitle(batch, y=1.02, fontsize=12)
    sns.despine()
    fig.tight_layout()
    savefig(fig, f"fraction_vs_Total.scatter_pc1.{batch.split()[0].replace('/', '_')}")
    plt.show()

## Sequencing depth: the slope expected from depth alone

The fraction libraries have 10 to 100 times fewer reads than Total. With fewer reads, weakly expressed genes get few or no counts, and their log2 CPM is pulled towards the pseudocount: the left tail of a shallow library is lifted, which by itself lowers the slope.

**Null curve from Total**: each siCtrl Total library is subsampled without replacement (multivariate hypergeometric draw from its counts on the genes) to a range of depths and compared with the sum of the other two siCtrl Total libraries; 10 subsamples per depth and split. This is the slope of a Total library of that depth, i.e. what depth alone gives. The **excess compression** of a library is its slope minus the null slope at its depth (interpolated in log depth): negative values mean more compression than depth explains.

**Depth-matched Total**, as in `pbody_RNA_Spang.SanityVmethod.ipynb`: the pooled siCtrl Total is subsampled to the depth of each fraction library (50 subsamples) and the slope of the library is recomputed against it; both axes then have the same Poisson noise.

Subsampling reproduces the random sampling of molecules at a lower depth. It does not reproduce how a library is made from less input material; that is the subject of the next sections.

In [ ]:
rng = np.random.default_rng(0)
counts_u = counts.loc[universe]
N_NULL, N_SUBSAMPLE = 10, 50

null_rows = []
depths = np.geomspace(2e4, 1e7, 16).astype(np.int64)
for l in total_ref:
    x = log2_cpm(counts[[r for r in total_ref if r != l]].sum(axis=1)).to_numpy()
    c = counts_u[l].to_numpy()
    for d in depths[depths < c.sum()]:
        for _ in range(N_NULL):
            sub = pd.Series(rng.multivariate_hypergeometric(c, d), index=universe)
            null_rows.append({"split": l, "depth": d, "pc1_slope": pc1_slope(x, log2_cpm(sub).to_numpy())})
null_df = pd.DataFrame(null_rows)
null_curve = null_df.groupby("depth")["pc1_slope"].agg(["mean", lambda s: s.quantile(0.025), lambda s: s.quantile(0.975)])
null_curve.columns = ["null_slope", "null_q025", "null_q975"]
save_tsv(null_curve.reset_index(), "depth_null.Total_subsampled_vs_Total.tsv")

def null_slope_at(depth):
    return np.interp(np.log10(depth), np.log10(null_curve.index), null_curve["null_slope"])

# depth-matched: pooled Total subsampled to the depth of each library
pooled_u = pooled_total.loc[universe].to_numpy()
dm_rows = []
for l in lib_slopes.index[lib_slopes["fraction"] != "Total"]:
    p = lib_values[l].to_numpy()
    s = [pc1_slope(log2_cpm(pd.Series(rng.multivariate_hypergeometric(pooled_u, int(qc.loc[l, "reads on genes"])), index=universe)).to_numpy(), p)
         for _ in range(N_SUBSAMPLE)]
    dm_rows.append({"library": l, "pc1_slope_depth_matched": np.mean(s), "depth_matched_q025": np.quantile(s, 0.025),
                    "depth_matched_q975": np.quantile(s, 0.975)})
lib_slopes = lib_slopes.drop(columns=[c for c in lib_slopes if c.startswith(("pc1_slope_depth", "depth_matched", "null_slope", "excess"))])
lib_slopes = lib_slopes.join(pd.DataFrame(dm_rows).set_index("library"))
lib_slopes["null_slope"] = null_slope_at(lib_slopes["reads_on_genes"])
lib_slopes["excess_compression"] = lib_slopes["pc1_slope"] - lib_slopes["null_slope"]
save_tsv(lib_slopes, "fraction_vs_Total.pc1_slope.per_library.tsv")

frac_slopes = lib_slopes.loc[lib_slopes["fraction"] != "Total"]
print(f"fraction libraries: median slope {frac_slopes['pc1_slope'].median():.3f}, median null slope at their depth "
      f"{frac_slopes['null_slope'].median():.3f}, median excess {frac_slopes['excess_compression'].median():.3f}; "
      f"{(frac_slopes['pc1_slope'] < frac_slopes['null_slope']).sum()} of {len(frac_slopes)} below the null")
display(frac_slopes[["batch", "fraction", "condition", "replicate", "reads_on_genes", "pc1_slope", "null_slope", "excess_compression",
                     "pc1_slope_depth_matched"]].round(3))

def legend_handles(df):
    return [plt.Line2D([], [], color=FRACTION_COLORS[f], marker="o", ls="", label=f) for f in FRACTION_ORDER if (df["fraction"] == f).any()] + \
           [plt.Line2D([], [], color="grey", marker=m, ls="", label=c) for c, m in CONDITION_MARKERS.items() if (df["condition"] == c).any()]

sns.set(font_scale=0.9, style="ticks")
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
for ax, ycol, title in ((axes[0], "pc1_slope", "observed (vs full Total); grey: Total subsampled to the depth"),
                        (axes[1], "pc1_slope_depth_matched", "vs Total subsampled to the depth of the library")):
    ax.fill_between(null_curve.index, null_curve["null_q025"], null_curve["null_q975"], color="lightgrey", lw=0)
    ax.plot(null_curve.index, null_curve["null_slope"], color="grey", lw=1.2, label="Total subsampled (null)")
    for _, r in lib_slopes.iterrows():
        if pd.isna(r[ycol]):
            continue
        ax.scatter(r["reads_on_genes"], r[ycol], color=FRACTION_COLORS[r["fraction"]], marker=CONDITION_MARKERS[r["condition"]], s=28, lw=0)
    ax.axhline(1, color="grey", lw=0.8, ls="--")
    ax.set(xscale="log", xlabel="reads of the library on the genes", ylabel="PC1 slope, library vs Total siCtrl")
    ax.set_title(title, fontsize=9)
axes[1].legend(handles=legend_handles(lib_slopes), fontsize=7, frameon=False, loc="upper left", bbox_to_anchor=(1.02, 1.0))
sns.despine()
fig.tight_layout()
savefig(fig, "fraction_vs_Total.pc1_slope_vs_depth")
plt.show()

#### Interpretation: slopes and depth

- **Reference spread.** Total libraries against Total have slopes 0.97–1.10 (siCtrl leave one out; wt and siCPSF6 vs siCtrl). This is the spread between replicates.
- **Depth does not compress here.** At the depth of the fraction libraries ($10^5$–$3 \cdot 10^6$ reads on the genes), a Total library subsampled to that depth has a slope of 1.02–1.08, slightly *above* 1: the extra Poisson noise widens the shallow library. Only below about 40,000 reads, where many genes have no counts, does depth alone push the slope below 1. Matching the depth of Total to each library lowers the slopes by at most 0.08 and does not change their order.
- **Golgi, PEX, Endo and TJ are compressed.** Golgi has slopes 0.81–0.92, PEX 0.81–0.88, Endo 0.84–0.93 and TJ 0.82–0.89, all below the depth null by 0.12–0.25, in both siCtrl and siCPSF5.
- **Org2 and ER are close to Total** (0.95–1.15, mostly within the null band), in all batches.
- **Free and TIS are widened, not compressed.** Free has slopes 1.07–1.25 in all three batches and TIS 1.15–1.21, above the null.
- **Caveat: the Total reference comes from a different preparation.** A Total library from another preparation (the 1000 ng libraries of the input control, below) has a slope of 1.12–1.17 against the HCT116 Total. A slope near 1.1 can therefore arise from the batch alone; the Free and TIS slopes are within that range.

## Library complexity

**Lander–Waterman estimate**, as in `pbody_RNA_Spang.SanityVmethod.ipynb`: the complexity $C$ of a library is the number of distinct template molecules it contains. With UMIs, a distinct molecule is a distinct combination of position and UMI (`umi_tools dedup`). If each of the $N$ sequenced reads comes from one of $C$ equally likely molecules, the expected number of distinct molecules observed is
$$U = C \cdot \left(1 - e^{-N/C}\right),$$
solved numerically for $C$, with $N$ and $U$ the mapped reads before and after deduplication (`master_read_tracking_stats.tsv`). For $N \ll C$, almost every read is a new molecule ($U \approx N$); the estimate of $C$ is then imprecise, because the fraction kept is close to 1. The model assumes every molecule equally likely to be sequenced; uneven PCR amplification violates it, so the values are a relative measure between libraries made with the same protocol.

Library properties against the slope, over the fraction libraries:
- reads on the genes;
- fraction of mapped reads kept by deduplication;
- estimated distinct molecules $C$;
- complexity relative to Total, $\log_2 C / \bar{C}_{Total}$, with $\bar{C}_{Total}$ the geometric mean of the three siCtrl Total libraries (no pairing of replicates);
- share of gene assignments on nuclear protein-coding genes (low when mitochondrial or other RNAs take a large share);
- nuclear contamination, $\log_2 R$ of MALAT1 (previous section).

Spearman correlations over all libraries, within batch (values centred on their batch mean) and within group (centred on the mean of the replicates of a fraction and condition). The same correlations are shown for the excess compression, which removes the part of the slope explained by depth.

The groups have two or three replicates, so the within-group correlation rests on few, nearly mirrored values per group. It is therefore also tested directly: in each group, is the slope of the most complex replicate higher than that of the least complex one (sign test over the groups)? Finally, a linear model of the slope on depth, complexity relative to Total and MALAT1 together (without and with the batch), over the fraction libraries, separates the three; depth and complexity are correlated, because $C$ is estimated from the same reads.

References:
- Lander ES, Waterman MS (1988). Genomic mapping by fingerprinting random clones: a mathematical analysis. *Genomics* 2:231–239.
- Smith T, Heger A, Sudbery I (2017). UMI-tools: modeling sequencing errors in Unique Molecular Identifiers to improve quantification accuracy. *Genome Research* 27:491–499.
- Picard `EstimateLibraryComplexity` (Broad Institute), https://broadinstitute.github.io/picard/

In [ ]:
def distinct_molecules(n_reads, n_unique):
    # Lander-Waterman: n_unique = C * (1 - exp(-n_reads / C)), solved for C
    if not (0 < n_unique < n_reads):
        return np.nan
    return brentq(lambda C: C * (1 - np.exp(-n_reads / C)) - n_unique, n_unique, n_unique * 1e6)

qc["distinct molecules"] = [distinct_molecules(n, u) for n, u in zip(qc["before dedup"], qc["after dedup"])]
C_total = np.exp(np.log(qc.loc[total_ref, "distinct molecules"]).mean())
qc["log2 complexity vs Total"] = np.log2(qc["distinct molecules"] / C_total)
save_tsv(qc, "QC.read_tracking_and_gene_assignments.tsv")
print(f"siCtrl Total: distinct molecules {', '.join(f'{v / 1e6:.1f} M' for v in qc.loc[total_ref, 'distinct molecules'])} "
      f"(geometric mean {C_total / 1e6:.1f} M)")

LIB_X = {"reads_on_genes": ("reads on the genes", "log"),
         "kept by dedup": ("fraction of mapped reads\nkept by UMI deduplication", "linear"),
         "distinct molecules": ("estimated distinct molecules", "log"),
         "log2 complexity vs Total": ("log2 distinct molecules,\nlibrary / Total siCtrl", "linear"),
         "share nuclear protein-coding": ("share of gene assignments on\nnuclear protein-coding genes", "linear"),
         "log2 R MALAT1": ("nuclear contamination,\nlog2 R MALAT1", "linear")}
cplx = frac_slopes.join(qc[[c for c in LIB_X if c in qc] + ["after dedup"]])

def correlations(df, xcols, y):
    rows = []
    for k in xcols:
        r, p = spearmanr(df[k], df[y], nan_policy="omit")
        out = {"x": k, "y": y, "n": int(df[[k, y]].notna().all(axis=1).sum()), "spearman": r, "p_value": p}
        for level, by in (("within_batch", "batch"), ("within_group", "group")):
            sub = df.loc[df[by].astype(str) != ""] if by == "group" else df
            sub = sub.loc[sub.groupby(by, observed=True)[y].transform("size") > 1]
            centred = sub[[k, y]] - sub.groupby(by, observed=True)[[k, y]].transform("mean")
            rc, pc = spearmanr(centred[k], centred[y], nan_policy="omit")
            out |= {f"spearman_{level}": rc, f"p_value_{level}": pc, f"n_{level}": len(sub)}
        rows.append(out)
    return pd.DataFrame(rows)

lib_corr = pd.concat([correlations(cplx, LIB_X, y) for y in ("pc1_slope", "excess_compression")], ignore_index=True)
save_tsv(lib_corr, "fraction_vs_Total.pc1_slope.library_properties.spearman.tsv")
save_tsv(cplx, "fraction_vs_Total.pc1_slope.library_properties.tsv")
display(lib_corr.round(4))

for y, ylabel in (("pc1_slope", "PC1 slope, library vs Total siCtrl"), ("excess_compression", "excess compression\n(slope - null slope at the depth)")):
    corr = lib_corr.loc[lib_corr["y"] == y].set_index("x")
    sns.set(font_scale=0.9, style="ticks")
    fig, axes = plt.subplots(1, len(LIB_X), figsize=(3.6 * len(LIB_X), 3.8), sharey=True)
    for ax, (k, (xlabel, xscale)) in zip(axes, LIB_X.items()):
        for _, r in cplx.iterrows():
            ax.scatter(r[k], r[y], color=FRACTION_COLORS[r["fraction"]], marker=CONDITION_MARKERS[r["condition"]], s=26, lw=0)
        ax.axhline(1 if y == "pc1_slope" else 0, color="grey", lw=0.8, ls="--")
        ax.set(xscale=xscale, xlabel=xlabel)
        c = corr.loc[k]
        ax.set_title(f"Spearman {c['spearman']:.2f}\nwithin batch {c['spearman_within_batch']:.2f}, within group {c['spearman_within_group']:.2f}", fontsize=8)
    axes[0].set_ylabel(ylabel)
    axes[-1].legend(handles=legend_handles(cplx), fontsize=7, frameon=False, loc="upper left", bbox_to_anchor=(1.02, 1.0))
    sns.despine()
    fig.tight_layout()
    savefig(fig, f"fraction_vs_Total.{y}.library_properties")
    plt.show()

# within each group: the more complex vs the less complex replicate (sign test over groups)
pair_rows = []
for g, sub in cplx.loc[cplx["group"] != ""].groupby("group"):
    if len(sub) < 2:
        continue
    hi, lo = sub["distinct molecules"].idxmax(), sub["distinct molecules"].idxmin()
    pair_rows.append({"group": g, "more_complex": hi, "less_complex": lo,
                      "log2_C_more_over_less": np.log2(sub.loc[hi, "distinct molecules"] / sub.loc[lo, "distinct molecules"]),
                      "delta_log2_R_MALAT1": sub.loc[hi, "log2 R MALAT1"] - sub.loc[lo, "log2 R MALAT1"],
                      "delta_pc1_slope": sub.loc[hi, "pc1_slope"] - sub.loc[lo, "pc1_slope"],
                      "delta_excess_compression": sub.loc[hi, "excess_compression"] - sub.loc[lo, "excess_compression"]})
pairs_df = pd.DataFrame(pair_rows)
save_tsv(pairs_df, "fraction_vs_Total.within_group.more_vs_less_complex.tsv")
display(pairs_df.drop(columns=["more_complex", "less_complex"]).round(3))
for col in ("delta_pc1_slope", "delta_excess_compression"):
    k = int((pairs_df[col] > 0).sum())
    print(f"{col}: higher in the more complex library in {k} of {len(pairs_df)} groups "
          f"(one-sided sign test p = {stats.binomtest(k, len(pairs_df), 0.5, alternative='greater').pvalue:.3f})")

# joint linear model over the fraction libraries: depth, complexity relative to Total and nuclear contamination together
import statsmodels.formula.api as smf
d = pd.DataFrame({"slope": cplx["pc1_slope"], "log10_reads": np.log10(cplx["reads_on_genes"]), "log2_C_vs_Total": cplx["log2 complexity vs Total"],
                  "log2_R_MALAT1": cplx["log2 R MALAT1"], "batch": cplx["batch"].astype(str)})
ols_rows = []
for name, formula in (("without batch", "slope ~ log10_reads + log2_C_vs_Total + log2_R_MALAT1"),
                      ("with batch", "slope ~ log10_reads + log2_C_vs_Total + log2_R_MALAT1 + C(batch)")):
    fit = smf.ols(formula, data=d).fit()
    for term in ("log10_reads", "log2_C_vs_Total", "log2_R_MALAT1"):
        ols_rows.append({"model": name, "term": term, "coefficient": fit.params[term], "ci_low": fit.conf_int().loc[term, 0],
                         "ci_high": fit.conf_int().loc[term, 1], "p_value": fit.pvalues[term], "R2_model": fit.rsquared, "n": int(fit.nobs)})
ols_df = pd.DataFrame(ols_rows)
save_tsv(ols_df, "fraction_vs_Total.pc1_slope.joint_model.tsv")
display(ols_df.round(4))

#### Interpretation: compression follows nuclear contamination, not the complexity of the fraction libraries

- **Complexity does not explain the slopes across fractions.** Over the 47 fraction libraries, the slope does not correlate with the estimated complexity (Spearman −0.06), with complexity relative to Total (−0.06), or with depth (0.02). The fraction libraries have 2 to 7 log2 fewer distinct molecules than Total. Among the least complex ones (−4 to −7 log2), the Free, TIS and most Org2 libraries are not compressed (slopes 0.97–1.25), while the Endo, TJ and PEX libraries of the same complexity are (0.81–0.93).
- **The slope follows nuclear contamination.** Spearman −0.82 with $\log_2 R$ of MALAT1 over all libraries and −0.84 within batch; 0.64 with the share of reads on nuclear protein-coding genes. In the joint model, MALAT1 lowers the slope by 0.045 per log2 (95% CI 0.034–0.056, p < 10⁻⁴), with depth and complexity in the model; depth and complexity have no significant effect (R² 0.62; 0.71 with the batch).
- **Within a fraction and condition, complexity has no consistent effect.** The most complex replicate has the higher slope in 9 of 18 groups (sign test p = 0.59): in 7 of 8 groups of the Golgi/PEX batch, but in 1 of 8 of the End/TJ batch. The within-group Spearman of complexity relative to Total with the slope is −0.30 (p = 0.05), opposite to a complexity effect, and −0.10 with the excess compression. MALAT1 does not correlate with the slope within groups either (0.03). The differences between replicates are small compared with those between fractions.
- **A possible mechanism (hypothesis).** The poly(A) RNA of the nucleus reflects synthesis more than steady-state levels. Stable, abundant mRNAs accumulate in the cytoplasm, so nuclear RNA has a narrower range over genes than whole-cell RNA. Mixing nuclear RNA into a fraction would compress it in proportion to the nuclear share, which is what Golgi, PEX, Endo and TJ show. A nuclear fraction library, or the share of intronic reads per library, would test this directly.
- **Caveat on the complexity estimates.** For the shallow fraction libraries, the fraction kept by deduplication is 0.80–0.96, where the Lander–Waterman estimate of $C$ is imprecise. Their complexity is also compared with a Total from another preparation.

## Pairwise comparisons: one library against one reference library

Above, every fraction library is compared with the **pooled** siCtrl Total: the counts of the three libraries are summed, then converted to CPM. Two points:
- **Normalization is not the issue.** Summing counts gives a CPM that sums to $10^6$ like any single library; it is not an average of CPMs. Even an average of log CPMs, which does not sum to $10^6$ (it is a geometric mean; the Sanity cohort averages used by fracSanity fall 0.01–0.11 log2 short), differs from a normalized profile mainly by a constant over genes, and a constant does not change a PC1 slope, which is computed on centred values.
- **Noise and weighting can be.** The pooled Total has about a third of the sampling noise of one library, and its libraries are weighted by their depth. Noise widens the axis it is on: on the reference axis it lowers the slope, on the fraction axis it raises it. Averaging (or pooling) also removes the replicate-to-replicate variation of the reference, which a single library keeps.

Here each analysed fraction library is compared with **each siCtrl Total library separately** (one slope per pair; mean, minimum and maximum over the three), the siCtrl Total libraries with each other (6 ordered pairs: the spread between single replicates), and a depth null for single libraries (each siCtrl Total library subsampled to a range of depths against each other single siCtrl Total library, 5 subsamples per depth). The correlations with the library properties are repeated with the pairwise slopes.

In [ ]:
# one fraction library against one siCtrl Total library
pair_rows = [{"library": l, "reference": t, "pc1_slope": pc1_slope(lib_values[t].to_numpy(), lib_values[l].to_numpy())}
             for l in analysed for t in total_ref]
pair_rows += [{"library": b, "reference": a, "pc1_slope": pc1_slope(lib_values[a].to_numpy(), lib_values[b].to_numpy())}
              for a in total_ref for b in total_ref if a != b]
pairwise = pd.DataFrame(pair_rows)
save_tsv(pairwise, "fraction_vs_Total.pc1_slope.pairwise.tsv")

# depth null for single libraries: one Total library subsampled, against another single Total library
null_pair_rows = []
for a in total_ref:
    for b in total_ref:
        if a == b:
            continue
        c = counts_u[b].to_numpy()
        for d in depths[depths < c.sum()]:
            for _ in range(5):
                sub = pd.Series(rng.multivariate_hypergeometric(c, d), index=universe)
                null_pair_rows.append({"reference": a, "subsampled": b, "depth": d, "pc1_slope": pc1_slope(lib_values[a].to_numpy(), log2_cpm(sub).to_numpy())})
null_pair = pd.DataFrame(null_pair_rows).groupby("depth")["pc1_slope"].agg(["mean", lambda s: s.quantile(0.025), lambda s: s.quantile(0.975)])
null_pair.columns = ["null_slope_pairwise", "q025", "q975"]
save_tsv(null_pair.reset_index(), "depth_null.single_Total_subsampled_vs_single_Total.tsv")

pw = pairwise.loc[pairwise["library"].isin(analysed)].groupby("library")["pc1_slope"].agg(pairwise_mean="mean", pairwise_min="min", pairwise_max="max")
pw["null_slope_pairwise"] = np.interp(np.log10(lib_slopes.loc[pw.index, "reads_on_genes"].astype(float)), np.log10(null_pair.index),
                                      null_pair["null_slope_pairwise"])
pw["excess_compression_pairwise"] = pw["pairwise_mean"] - pw["null_slope_pairwise"]
cplx_pw = cplx.join(pw)
cplx_pw["pairwise_minus_pooled"] = cplx_pw["pairwise_mean"] - cplx_pw["pc1_slope"]
save_tsv(cplx_pw, "fraction_vs_Total.pc1_slope.pairwise_vs_pooled.tsv")
tt = pairwise.loc[pairwise["library"].isin(total_ref), "pc1_slope"]
print(f"siCtrl Total vs siCtrl Total, single libraries: slopes {tt.min():.3f}-{tt.max():.3f} (6 ordered pairs)")
print(f"fraction libraries: pairwise mean - pooled slope: median {cplx_pw['pairwise_minus_pooled'].median():+.3f}, "
      f"range {cplx_pw['pairwise_minus_pooled'].min():+.3f} to {cplx_pw['pairwise_minus_pooled'].max():+.3f}; "
      f"Spearman pairwise vs pooled {spearmanr(cplx_pw['pairwise_mean'], cplx_pw['pc1_slope'])[0]:.3f}")
display(cplx_pw.groupby(["batch", "fraction"], observed=True)[["pc1_slope", "pairwise_mean", "pairwise_min", "pairwise_max"]].agg(["min", "max"]).round(3))

pw_corr = pd.concat([correlations(cplx_pw, LIB_X, y) for y in ("pairwise_mean", "excess_compression_pairwise")], ignore_index=True)
save_tsv(pw_corr, "fraction_vs_Total.pc1_slope.pairwise.library_properties.spearman.tsv")
display(pd.concat({"pooled reference": lib_corr.set_index(["y", "x"]), "pairwise": pw_corr.set_index(["y", "x"])})
        [["n", "spearman", "p_value", "spearman_within_batch", "spearman_within_group"]].round(4))

sns.set(font_scale=0.9, style="ticks")
fig, axes = plt.subplots(1, 4, figsize=(19, 4.3))
ax = axes[0]
for l, r in cplx_pw.iterrows():
    ax.errorbar(r["pc1_slope"], r["pairwise_mean"], yerr=[[r["pairwise_mean"] - r["pairwise_min"]], [r["pairwise_max"] - r["pairwise_mean"]]],
                fmt=CONDITION_MARKERS[r["condition"]], color=FRACTION_COLORS[r["fraction"]], ms=5, elinewidth=0.6, capsize=0)
lo, hi = 0.7, 1.4
ax.plot([lo, hi], [lo, hi], color="grey", ls="--", lw=0.8)
ax.set(xlim=(lo, hi), ylim=(lo, hi), xlabel="PC1 slope vs pooled siCtrl Total", ylabel="PC1 slope vs single siCtrl Total\n(mean, range over 3)")
ax = axes[1]
ax.fill_between(null_curve.index, null_curve["null_q025"], null_curve["null_q975"], color="lightgrey", lw=0)
ax.plot(null_curve.index, null_curve["null_slope"], color="grey", lw=1.2, label="null: vs pooled Total")
ax.fill_between(null_pair.index, null_pair["q025"], null_pair["q975"], color="tab:blue", alpha=0.15, lw=0)
ax.plot(null_pair.index, null_pair["null_slope_pairwise"], color="tab:blue", lw=1.2, label="null: vs single Total")
for l, r in cplx_pw.iterrows():
    ax.scatter(r["reads_on_genes"], r["pairwise_mean"], color=FRACTION_COLORS[r["fraction"]], marker=CONDITION_MARKERS[r["condition"]], s=26, lw=0)
ax.axhline(1, color="grey", lw=0.8, ls="--")
ax.set(xscale="log", xlabel="reads on the genes", ylabel="PC1 slope vs single Total (mean)")
ax.legend(fontsize=7, frameon=False)
for ax, k in zip(axes[2:], ("log2 R MALAT1", "log2 complexity vs Total")):
    for l, r in cplx_pw.iterrows():
        ax.scatter(r[k], r["pairwise_mean"], color=FRACTION_COLORS[r["fraction"]], marker=CONDITION_MARKERS[r["condition"]], s=26, lw=0)
    c = pw_corr.loc[(pw_corr["y"] == "pairwise_mean") & (pw_corr["x"] == k)].iloc[0]
    ax.axhline(1, color="grey", lw=0.8, ls="--")
    ax.set(xlabel=LIB_X[k][0], ylabel="PC1 slope vs single Total (mean)",
           title=f"Spearman {c['spearman']:.2f}, within batch {c['spearman_within_batch']:.2f}, within group {c['spearman_within_group']:.2f}")
axes[-1].legend(handles=legend_handles(cplx_pw), fontsize=7, frameon=False, loc="upper left", bbox_to_anchor=(1.02, 1.0))
sns.despine(); fig.tight_layout()
savefig(fig, "fraction_vs_Total.pc1_slope.pairwise")
plt.show()

#### Interpretation: pairwise comparisons

- **Single Total libraries against each other** have slopes 0.92–1.09 (6 ordered pairs), a slightly wider spread than one library against the other two pooled (0.97–1.10): the reference itself varies between replicates by about ±0.08.
- **Every fraction library has a slope 0.022–0.030 lower against a single Total library than against the pooled Total** (median −0.025). The shift is the same for all libraries: a single library adds sampling noise on the Total axis, which widens it and lowers the slope. The ordering of the libraries is unchanged (Spearman of pairwise vs pooled slopes 1.00), and so are the fraction ranges (Golgi 0.78–0.90, PEX 0.79–0.86, Endo 0.81–0.91, TJ 0.79–0.86; Free 1.05–1.22, TIS 1.13–1.19).
- **The associations do not change.** Pairwise slope vs nuclear contamination (log2 R MALAT1): Spearman −0.83 (pooled −0.82); vs complexity −0.05 (pooled −0.06); vs depth 0.03 (pooled 0.02).
- **Normalization of averages.** The pooled reference sums counts, so it is normalized like a single library; a constant offset over genes, which is what averaging log values produces, cannot change a PC1 slope. The pairwise analysis confirms that pooling the reference does not create or hide the compression; it only shifts all slopes up by ~0.025.

## Control: Total RNA library from 1000 ng vs 100 ng of input

Two libraries from the same Total RNA, one from 1000 ng and one from a 100 ng aliquot, prepared and sequenced together (siCtrl: barcode01 and barcode02; siCPSF5: barcode03 and barcode04). The RNA composition is the same; only the input amount, and with it the complexity of the library, differs. If a lower complexity alone compresses a library, the 100 ng library plotted against the 1000 ng library has a PC1 slope below 1, and below what its depth explains.

Per pair:
- **direct comparison**: log2 CPM of 100 ng (y) vs 1000 ng (x), same genes as above;
- **depth-matched**: the deeper library of the pair is subsampled to the reads of the other (50 subsamples), so that both have the same number of reads; the slope is recomputed;
- **against Total siCtrl** (the reference of the fractions): slope of the 1000 ng and 100 ng libraries, and of the 1000 ng library subsampled to the depth of the 100 ng library (50 subsamples). If the 100 ng library were a random subset of the molecules of the 1000 ng library, its slope would match the subsampled 1000 ng library.

The pairs are then placed on the slope vs complexity plot of the fractions.

In [ ]:
CONTROL_PAIRS = {c: (libs.index[(libs["fraction"] == "Total 1000 ng") & (libs["condition"] == c)][0],
                     libs.index[(libs["fraction"] == "Total 100 ng") & (libs["condition"] == c)][0]) for c in ("siCtrl", "siCPSF5")}
ctrl_rows, ctrl_sub = [], {}
for cond, (hi, lo) in CONTROL_PAIRS.items():
    x, y = lib_values[hi].to_numpy(), lib_values[lo].to_numpy()
    n_hi, n_lo = int(qc.loc[hi, "reads on genes"]), int(qc.loc[lo, "reads on genes"])
    deeper, other, n_match = (hi, lo, n_lo) if n_hi > n_lo else (lo, hi, n_hi)
    dm = []
    for _ in range(N_SUBSAMPLE):
        sub = log2_cpm(pd.Series(rng.multivariate_hypergeometric(counts_u[deeper].to_numpy(), n_match), index=universe)).to_numpy()
        dm.append(pc1_slope(sub, y) if deeper == hi else pc1_slope(x, sub))
    # vs Total siCtrl: 1000 ng subsampled to the depth of the 100 ng library
    hi_sub = [pc1_slope(t_ref, log2_cpm(pd.Series(rng.multivariate_hypergeometric(counts_u[hi].to_numpy(), min(n_lo, n_hi)), index=universe)).to_numpy())
              for _ in range(N_SUBSAMPLE)]
    ctrl_sub[cond] = (dm, hi_sub)
    ctrl_rows.append({"condition": cond, "library_1000ng": hi, "library_100ng": lo, "reads_on_genes_1000ng": n_hi, "reads_on_genes_100ng": n_lo,
                      "kept_by_dedup_1000ng": qc.loc[hi, "kept by dedup"], "kept_by_dedup_100ng": qc.loc[lo, "kept by dedup"],
                      "distinct_molecules_1000ng": qc.loc[hi, "distinct molecules"], "distinct_molecules_100ng": qc.loc[lo, "distinct molecules"],
                      "log2_complexity_100_over_1000": np.log2(qc.loc[lo, "distinct molecules"] / qc.loc[hi, "distinct molecules"]),
                      "pc1_slope_100_vs_1000": pc1_slope(x, y), "pearson_r_100_vs_1000": np.corrcoef(x, y)[0, 1],
                      "depth_matched_mean": np.mean(dm), "depth_matched_q025": np.quantile(dm, 0.025), "depth_matched_q975": np.quantile(dm, 0.975),
                      "subsampled": "1000 ng" if deeper == hi else "100 ng",
                      "null_slope_at_100ng_depth": null_slope_at(n_lo),
                      "pc1_slope_1000ng_vs_Total": pc1_slope(t_ref, x), "pc1_slope_100ng_vs_Total": pc1_slope(t_ref, y),
                      "pc1_slope_1000ng_subsampled_vs_Total": np.mean(hi_sub),
                      "p_100ng_below_subsampled_1000ng": (np.sum(np.array(hi_sub) <= pc1_slope(t_ref, y)) + 1) / (len(hi_sub) + 1)})
ctrl_df = pd.DataFrame(ctrl_rows).set_index("condition")
save_tsv(ctrl_df.reset_index(), "input_amount_control.1000ng_vs_100ng.tsv")
display(ctrl_df.T)

sns.set(font_scale=0.9, style="ticks")
fig, axes = plt.subplots(1, len(CONTROL_PAIRS) + 1, figsize=(4.2 * (len(CONTROL_PAIRS) + 1), 4.2))
for ax, (cond, (hi, lo)) in zip(axes, CONTROL_PAIRS.items()):
    x, y = lib_values[hi].to_numpy(), lib_values[lo].to_numpy()
    r = ctrl_df.loc[cond]
    ax.scatter(x, y, s=1.5, alpha=0.25, color=FRACTION_COLORS["Total 100 ng"], lw=0, rasterized=True)
    ax.plot(lims, y.mean() + (lims - x.mean()), color="grey", lw=0.8, ls="--")
    ax.plot(lims, y.mean() + r["pc1_slope_100_vs_1000"] * (lims - x.mean()), color="black", lw=1.2)
    ax.text(0.04, 0.96, f"PC1 slope {r['pc1_slope_100_vs_1000']:.3f}\ndepth-matched {r['depth_matched_mean']:.3f} "
            f"[{r['depth_matched_q025']:.3f}, {r['depth_matched_q975']:.3f}]\nlog2 C 100/1000 = {r['log2_complexity_100_over_1000']:.2f}",
            transform=ax.transAxes, va="top", fontsize=7.5)
    ax.set(xlim=lims, ylim=lims, aspect="equal", xlabel="log2 CPM, 1000 ng", ylabel="log2 CPM, 100 ng", title=f"{cond} Total")
ax = axes[-1]
for k, (cond, r) in enumerate(ctrl_df.iterrows()):
    vals = [r["pc1_slope_1000ng_vs_Total"], r["pc1_slope_1000ng_subsampled_vs_Total"], r["pc1_slope_100ng_vs_Total"]]
    ax.plot(np.arange(3) + 0.08 * k, vals, marker=CONDITION_MARKERS[cond], color="black", lw=0.8, label=cond)
ax.set_xticks(range(3))
ax.set_xticklabels(["1000 ng", "1000 ng subsampled\nto 100 ng depth", "100 ng"], fontsize=8)
ax.set(ylabel="PC1 slope vs Total siCtrl (HCT116)")
ax.axhline(1, color="grey", lw=0.8, ls="--")
ax.legend(fontsize=7, frameon=False)
sns.despine()
fig.tight_layout()
savefig(fig, "input_amount_control.1000ng_vs_100ng")
plt.show()

# the control pairs on the slope vs complexity plots of the fractions: x = complexity of the library relative to its reference
# (fractions: log2 C / C of Total siCtrl; control: log2 C 100 ng / C 1000 ng), y = slope against the reference
sns.set(font_scale=0.9, style="ticks")
fig, ax = plt.subplots(figsize=(6.2, 4.4))
for _, r in cplx.iterrows():
    ax.scatter(r["log2 complexity vs Total"], r["pc1_slope"], color=FRACTION_COLORS[r["fraction"]], marker=CONDITION_MARKERS[r["condition"]], s=26, lw=0)
for cond, r in ctrl_df.iterrows():
    ax.scatter(r["log2_complexity_100_over_1000"], r["pc1_slope_100_vs_1000"], facecolor="none", edgecolor="black", marker=CONDITION_MARKERS[cond], s=90, lw=1.5)
    ax.annotate(f"100 vs 1000 ng, {cond}", (r["log2_complexity_100_over_1000"], r["pc1_slope_100_vs_1000"]), fontsize=7, xytext=(5, -10), textcoords="offset points")
ax.axhline(1, color="grey", lw=0.8, ls="--")
ax.axvline(0, color="grey", lw=0.8)
ax.set(xlabel="log2 distinct molecules, library / reference", ylabel="PC1 slope, library vs reference")
ax.set_title("fractions vs Total siCtrl (filled); 100 ng vs 1000 ng (open)", fontsize=9)
ax.legend(handles=legend_handles(cplx), fontsize=7, frameon=False, loc="upper left", bbox_to_anchor=(1.02, 1.0))
sns.despine()
fig.tight_layout()
savefig(fig, "slope_vs_complexity.fractions_and_input_control")
plt.show()

#### Interpretation: the input amount control

- **The 100 ng libraries have fewer distinct molecules at the same depth.** They have 3.4–3.5 M reads on the genes against 3.8 M for 1000 ng, but deduplication keeps only 0.71 and 0.68 of their reads (0.92 and 0.93 at 1000 ng). Their estimated complexity is 3.5–4.3 times lower (log2 −1.81 for siCtrl, −2.09 for siCPSF5).
- **They are compressed.** The slope of 100 ng against 1000 ng is 0.911 (siCtrl) and 0.943 (siCPSF5), with Pearson r 0.98. The RNA is the same, so this is outside the 0.97–1.10 spread between biological replicates of Total. Matching the depth leaves the slopes unchanged (0.911 and 0.943), so the compression is not a depth effect.
- **The same holds against the HCT116 Total.** The 100 ng libraries have slopes 1.06 and 1.05, against 1.17 and 1.12 for 1000 ng. The 1000 ng library subsampled to the depth of the 100 ng library keeps its slope (1.18 and 1.12; 100 ng is below all 50 subsamples, p = 0.02, the smallest value possible with 50 subsamples). The 100 ng library is therefore not a random subset of the molecules of the 1000 ng library: a library made from less input is compressed beyond what sampling explains.
- **Size of the effect.** The slope drops by about 0.03–0.05 per log2 of complexity ((1 − 0.911)/1.81 = 0.049; (1 − 0.943)/2.09 = 0.027), similar to the 0.054 per log2 found for the P-body pull-downs in `pbody_RNA_Spang.SanityVmethod.ipynb`.
- **Conclusion.** This validates a causal link between lower library complexity and compression. Its size, however, does not explain the fractions: at that rate, the 2–7 log2 lower complexity of the fraction libraries would compress all of them to slopes of 0.65–0.95, while Free and TIS are above 1 and the compression follows nuclear contamination instead (previous section).

## Summary

- **Data.** 57 fraction libraries (Free, Org2, ER, TIS, Golgi, PEX, Endo, TJ in siCtrl and siCPSF5) were compared with the pooled siCtrl HCT116 Total; 47 have at least 100,000 reads on the 4,578 genes used.
- **Compression.** The Golgi, PEX, Endo and TJ fractions are compressed against Total (PC1 slopes 0.81–0.93), well below what their depth explains. Org2 and ER are close to Total. Free and TIS are not compressed (slopes 1.07–1.25), within the range a Total library from another preparation shows against the HCT116 Total (1.12–1.17).
- **Nuclear contamination** (MALAT1 per mRNA, relative to whole cells) is low in Free and TIS (≤ 0.45), 0.6–2.7 in Org2 and ER, 1.4–2.0 in Endo, 2.8–4.5 in Golgi, 3.6–7.1 in TJ and 4.6–8.2 in PEX. With 5–10% of whole-cell poly(A) RNA in the nucleus, 35–80% of the RNA of the siCtrl PEX and TJ libraries would be nuclear.
- **The compression of the fractions follows their nuclear contamination** (Spearman −0.82; −0.045 slope per log2 MALAT1 in a joint model), not their library complexity or depth. Between replicates of the same fraction there is no consistent relation with complexity (higher slope in the more complex replicate in 9 of 18 groups).
- **The input amount control establishes that complexity can cause compression.** A Total library from 100 ng of input has 3.5–4.3 times fewer distinct molecules than one from 1000 ng of the same RNA, and is compressed against it (slopes 0.91 and 0.94), independently of depth (about 0.03–0.05 per log2 complexity).
- **Next steps:**
  - a Total library prepared and sequenced with each fraction batch, to remove the batch term from the slopes;
  - a nuclear fraction, or the intronic read share per library, to test whether nuclear RNA explains the compression of Golgi, PEX, Endo and TJ;
  - deeper sequencing of the fraction libraries, for better complexity estimates.